In [1]:
# ==========================================
# 1. Import Libraries
# ==========================================

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from xgboost import XGBClassifier

# ==========================================
# 2. Load Dataset
# ==========================================

df = pd.read_csv("/content/ecommerce_customer_support_dataset_after_eda.csv")

# Remove Ticket ID
df.drop("Ticket_ID", axis=1, inplace=True)

# ==========================================
# 3. Features and Target
# ==========================================

X = df.drop("Misrouted", axis=1)
y = df["Misrouted"]

# Convert Target into Numeric
y = y.map({
    "Yes": 1,
    "No": 0
})

# ==========================================
# 4. Feature Groups
# ==========================================

text_feature = "Customer_Query"

categorical_features = [
    "Category",
    "Department",
    "Priority",
    "Sentiment"
]

numeric_features = [
    "Confidence_Score",
    "Escalation_Count",
    "Previous_Transfers",
    "Resolution_Time_Hours"
]

# ==========================================
# 5. Preprocessing
# ==========================================

preprocessor = ColumnTransformer(

    transformers=[

        (
            "text",
            TfidfVectorizer(stop_words="english"),
            text_feature
        ),

        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        ),

        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_features
        )

    ]

)

# ==========================================
# 6. XGBoost Model
# ==========================================

xgb = XGBClassifier(

    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"

)

pipeline = Pipeline([

    ("preprocessor", preprocessor),

    ("classifier", xgb)

])

# ==========================================
# 7. Train Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y

)

# ==========================================
# 8. Train Model
# ==========================================

pipeline.fit(X_train, y_train)

# ==========================================
# 9. Prediction
# ==========================================

y_pred = pipeline.predict(X_test)

y_prob = pipeline.predict_proba(X_test)[:,1]

# ==========================================
# 10. Accuracy
# ==========================================

accuracy = accuracy_score(y_test, y_pred)

print("="*60)
print("Accuracy")
print("="*60)
print(f"Accuracy : {accuracy:.4f}")
print(f"Accuracy Percentage : {accuracy*100:.2f}%")

# ==========================================
# 11. Classification Report
# ==========================================

print("\n")
print("="*60)
print("Classification Report")
print("="*60)

print(classification_report(y_test, y_pred))

# ==========================================
# 12. Confusion Matrix
# ==========================================

print("\n")
print("="*60)
print("Confusion Matrix")
print("="*60)

print(confusion_matrix(y_test, y_pred))

# ==========================================
# 13. ROC-AUC Score
# ==========================================

roc = roc_auc_score(y_test, y_prob)

print("\n")
print("="*60)
print("ROC-AUC Score")
print("="*60)

print(f"ROC-AUC : {roc:.4f}")

# ==========================================
# 14. Save Model
# ==========================================

joblib.dump(pipeline, "misrouting_xgboost.pkl")

print("\nModel Saved Successfully!")

Accuracy
Accuracy : 1.0000
Accuracy Percentage : 100.00%


Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       222
           1       1.00      1.00      1.00      1778

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



Confusion Matrix
[[ 222    0]
 [   0 1778]]


ROC-AUC Score
ROC-AUC : 1.0000

Model Saved Successfully!
